In [3]:
import os
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay
)

print("MLflow version:", mlflow.__version__)

MLflow version: 3.16.1


In [4]:
TRACKING_URI = "sqlite:///mlruns.db"

mlflow.set_tracking_uri(TRACKING_URI)

print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: sqlite:///mlruns.db


In [5]:
data = load_breast_cancer()

x = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Features shape:", x.shape)
print("Target shape:", y.shape)

Features shape: (569, 30)
Target shape: (569,)


In [6]:
x.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [7]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

#data scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
X_train.shape

(455, 30)

In [9]:
baseline_model = LogisticRegression(C=1.0, max_iter=1000)

baseline_model.fit(X_train_scaled, y_train)

baseline_pred = baseline_model.predict(X_test_scaled)

baseline_pred_proba = baseline_model.predict_proba(X_test_scaled)[:, 1]

baseline_accuracy = accuracy_score(y_test, baseline_pred)

baseline_precision = precision_score(y_test, baseline_pred)

baseline_recall = recall_score(y_test, baseline_pred)

baseline_f1 = f1_score(y_test, baseline_pred)

baseline_roc_auc = roc_auc_score(y_test, baseline_pred)

In [10]:
print("Baseline Model Performance:")
print(f"Accuracy: {baseline_accuracy}")
print(f"Precision: {baseline_precision}")
print(f"Recall: {baseline_recall}")
print(f"F1-Score: {baseline_f1}")
print(f"ROC AUC: {baseline_roc_auc}")

Baseline Model Performance:
Accuracy: 0.9736842105263158
Precision: 0.9722222222222222
Recall: 0.9859154929577465
F1-Score: 0.9790209790209791
ROC AUC: 0.969701932525385


In [11]:
Experiment_name = "Breast_Cancer_Classification"

experiment = mlflow.set_experiment(Experiment_name)

print("Experiment ID:", experiment.experiment_id)

print("Experiment Name:", experiment.name)

2026/09/19 11:24:10 INFO mlflow.tracking.fluent: Experiment with name 'Breast_Cancer_Classification' does not exist. Creating a new experiment.


Experiment ID: 1
Experiment Name: Breast_Cancer_Classification


In [12]:
with mlflow.start_run(run_name="Baseline_Logistic_Regression") as run:
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("max_iter", 1000)

    mlflow.log_metric("accuracy", baseline_accuracy)
    mlflow.log_metric("precision", baseline_precision)
    mlflow.log_metric("recall", baseline_recall)
    mlflow.log_metric("f1_score", baseline_f1)
    mlflow.log_metric("roc_auc", baseline_roc_auc)

    mlflow.sklearn.log_model(baseline_model, "baseline_model")

    print("Run ID:", run.info.run_id)

2026/09/19 11:53:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: 57aa756833fc421ea7385a2ae9f889e8


In [13]:
with mlflow.start_run(run_name="Baseline_Logistic_Regression_logTags") as run:
    mlflow.set_tag("model_type", "Logistic Regression")
    mlflow.set_tag("model family", "linear")
    mlflow.set_tag("dataset", "breast_cancer")
    mlflow.set_tag("stage", "development")

    mlflow.log_param("c", 1.0)
    mlflow.log_param("max_iter", 1000)

    mlflow.log_metric("accuracy", baseline_accuracy)
    mlflow.log_metric("precision", baseline_precision)
    mlflow.log_metric("recall", baseline_recall)
    mlflow.log_metric("f1_score", baseline_f1)
    mlflow.log_metric("roc_auc", baseline_roc_auc)

    mlflow.sklearn.log_model(baseline_model, "baseline_model")

    print("Run ID:", run.info.run_id)

2026/09/19 12:03:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: 32f75384ef144fa4b15be6fac59d5054


In [15]:
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]

for c in C_values:
    with mlflow.start_run(run_name=f"Logistic_Regression_C_{c}") as run:
        model = LogisticRegression(C=c, max_iter=1000)
        model.fit(X_train_scaled, y_train)

        pred = model.predict(X_test_scaled)
        pred_proba = model.predict_proba(X_test_scaled)[:, 1]

        accuracy = accuracy_score(y_test, pred)
        precision = precision_score(y_test, pred)
        recall = recall_score(y_test, pred)
        f1 = f1_score(y_test, pred)
        roc_auc = roc_auc_score(y_test, pred)

        mlflow.log_param("model_type", "Logistic Regression")
        mlflow.log_param("C", c)
        mlflow.log_param("max_iter", 1000)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", roc_auc)

       

        print(f"Run ID for C={c}:", run.info.run_id)

Run ID for C=0.01: be4cb92308ef41ceb35e3e7482c34774
Run ID for C=0.1: 7df497acbbcc4c088dc797805fd10278
Run ID for C=1.0: 66ce89f0d9b549c8a34b9b3ff646076b
Run ID for C=10.0: 682945b395bf425c85ea599e685b842c
Run ID for C=100.0: d9340ba2b056453583731f0745548225


In [16]:
with mlflow.start_run(run_name="artifact_confusion_matrix") as run:
    model = LogisticRegression(C=0.1, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    pred = model.predict(X_test_scaled)

    cm_display = ConfusionMatrixDisplay.from_estimator(model, X_test_scaled, y_test)
    plt.title("Confusion Matrix")
    plt.savefig("confusion_matrix.png")
    plt.close()

    mlflow.log_artifact("confusion_matrix.png")

    print("Run ID:", run.info.run_id)

Run ID: 5d184ff9a077425c84f9d58840fd33b7


In [17]:
with mlflow.start_run(run_name="classification_reporttxt_artifact") as run:
    model = LogisticRegression(C=0.1, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    pred = model.predict(X_test_scaled)

    report = classification_report(y_test, pred, output_dict=True)
    with open("classification_report.txt", "w") as f:
        json.dump(report, f)

    mlflow.log_artifact("confusion_matrix.png")
    mlflow.log_artifact("classification_report.txt")

    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("C", 0.1)
    mlflow.log_param("max_iter", 1000)

    print("Run ID:", run.info.run_id)

Run ID: 604117448cd84b4f842e5242dd06e48e
